In [1]:
#! Import dependencies and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import math
import os
import random
import sys
import time

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset
from torchvision import datasets

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if DEVICE.type != 'cuda':
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before training Task 4A.')

print('PyTorch:', torch.__version__, '| device:', torch.cuda.get_device_name(0))

Mounted at /content/drive
PyTorch: 2.11.0+cu128 | device: Tesla T4


In [2]:
#! Fix assignment hyperparameters and Task 4-only paths
ROOT = Path('/content/drive/MyDrive/pa1-beyond-iid')

TASK = ROOT / 'task4'
CODE_DIR = TASK / 'code'
DATA_DIR = TASK / 'data' / 'cifar10'
SPLIT_DIR = TASK / 'splits'
CONFIG_DIR = TASK / 'configs'
CKPT_DIR = TASK / 'checkpoints' / 'vanilla'
RESULT_DIR = TASK / 'results' / 'vanilla'
CACHE_DIR = TASK / 'cache' / 'vanilla'

for directory in [CODE_DIR, DATA_DIR, SPLIT_DIR, CONFIG_DIR, CKPT_DIR, RESULT_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 6304
EPOCHS = 100
BATCH_SIZE = 128
NUM_WORKERS = 2
LR, MOMENTUM, WEIGHT_DECAY = 0.1, 0.9, 5e-4

SPLIT_FILE = SPLIT_DIR / 'cifar10_stratified_90_10_seed6304.npz'
CONFIG_FILE = CONFIG_DIR / 'vanilla.json'
INITIAL_PATH = TASK / 'checkpoints' / 'shared_random_resnet18_seed6304.pt'
BEST_PATH = CKPT_DIR / 'best.pt'
LAST_PATH = CKPT_DIR / 'last.pt'
HISTORY_FILE = RESULT_DIR / 'history.csv'
CORE_FILE = CODE_DIR / 'cifar_osr_core_v1.py'

CONFIG = {
    'experiment': 'task4_vanilla', 'seed': SEED, 'dataset': 'CIFAR-10 official train only',
    'split': 'stratified 90/10', 'train_size': 45000, 'val_size': 5000,
    'architecture': 'torchvision resnet18 random CIFAR 3x3 stride1, identity maxpool, 10 classes',
    'input_shape': [3, 32, 32], 'normalization_mean': [0.4914, 0.4822, 0.4465],
    'normalization_std': [0.2470, 0.2435, 0.2616],
    'augmentation': 'RandomCrop(32,padding=4), RandomHorizontalFlip()',
    'optimizer': 'SGD', 'learning_rate': LR, 'momentum': MOMENTUM,
    'weight_decay': WEIGHT_DECAY, 'scheduler': 'CosineAnnealingLR T_max=100 eta_min=0',
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'selection': 'highest CIFAR-10 validation accuracy',
    'precision': 'float32', 'shared_code_version': 'cifar_osr_core_v1',
}

CONFIG_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()

if CONFIG_FILE.exists():
    if json.loads(CONFIG_FILE.read_text()) != CONFIG:
        raise RuntimeError('Existing vanilla configuration differs. Do not resume incompatible training.')
else:
    CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2) + '\n')

print('Task root:', TASK)
print('Configuration SHA256:', CONFIG_SHA)

Task root: /content/drive/MyDrive/pa1-beyond-iid/task4
Configuration SHA256: c55c3364f186960694dd17a7d2e22e506292ace9bf028df002efee24b6f9e156


In [3]:
#! Install the Task 4-only shared module without overwriting differing code
CORE_CODE = "#! Task 4 shared V1: isolated from Tasks 1-3 and stable for GCSC/PROSER\nimport os\nimport random\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torchvision import models, transforms\n\nCIFAR_MEAN = (0.4914, 0.4822, 0.4465)\nCIFAR_STD = (0.2470, 0.2435, 0.2616)\n\n\ndef seed_all(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef make_resnet18(num_classes=10):\n    #! Random initialization; CIFAR stem does not use the ImageNet 7x7 stem or max-pool.\n    model = models.resnet18(weights=None)\n    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)\n    model.maxpool = nn.Identity()\n    model.fc = nn.Linear(model.fc.in_features, num_classes)\n    return model\n\n\ndef forward_features_logits(model, images):\n    #! Exact ResNet-18 forward with accessible 512-dimensional pooled representation.\n    x = model.conv1(images)\n    x = model.bn1(x)\n    x = model.relu(x)\n    x = model.maxpool(x)\n    x = model.layer1(x)\n    x = model.layer2(x)\n    x = model.layer3(x)\n    x = model.layer4(x)\n    features = torch.flatten(model.avgpool(x), 1)\n    return features, model.fc(features)\n\n\ndef training_transform(gcsc=False):\n    ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip()]\n    if gcsc:\n        #! The sole augmentation difference in GCSC, before ToTensor and Normalize.\n        ops.append(transforms.RandAugment(num_ops=2, magnitude=9))\n    ops.extend([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])\n    return transforms.Compose(ops)\n\n\ndef evaluation_transform():\n    return transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])\n\n\ndef atomic_torch_save(payload, path):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + '.tmp')\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef extract_known_outputs(model, loader, device):\n    #! Sequential, unaugmented known-only loader; index order comes from its Subset.\n    model.eval()\n    features, logits, labels = [], [], []\n    with torch.inference_mode():\n        for images, targets in loader:\n            f, z = forward_features_logits(model, images.to(device, non_blocking=True))\n            features.append(f.float().cpu().numpy())\n            logits.append(z.float().cpu().numpy())\n            labels.append(targets.numpy())\n    return (np.concatenate(features), np.concatenate(logits), np.concatenate(labels))\n"

if CORE_FILE.exists():
    if CORE_FILE.read_text() != CORE_CODE:
        raise RuntimeError('Task 4 shared module differs; inspect ' + str(CORE_FILE) + ' before proceeding.')
    print('Reusing existing Task 4 module.')
else:
    CORE_FILE.write_text(CORE_CODE)
    print('Created:', CORE_FILE)

sys.path.insert(0, str(CODE_DIR)) if str(CODE_DIR) not in sys.path else None

from cifar_osr_core_v1 import (
    seed_all, make_resnet18, forward_features_logits, training_transform,
    evaluation_transform, atomic_torch_save, extract_known_outputs,
)

if (training_transform(False).transforms[0].padding != 4
        or len(training_transform(False).transforms) != 4
        or len(training_transform(True).transforms) != 5):
    raise RuntimeError('Training transforms do not have the expected crop/flip/augmentation order.')

print('Core module + transforms: PASS')

Created: /content/drive/MyDrive/pa1-beyond-iid/task4/code/cifar_osr_core_v1.py
Core module + transforms: PASS


In [4]:
#! Build exact CIFAR-10 train/validation indices, with no test/unknown access
seed_all(SEED)

raw_train = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True)
targets = np.asarray(raw_train.targets, dtype=np.int64)

if len(targets) != 50000 or sorted(np.unique(targets).tolist()) != list(range(10)):
    raise RuntimeError('The official CIFAR-10 training partition is not as expected.')

if SPLIT_FILE.exists():
    with np.load(SPLIT_FILE) as split:
        train_idx = split['train_idx'].astype(np.int64)
        val_idx = split['val_idx'].astype(np.int64)
else:
    rng = np.random.default_rng(SEED)
    train_groups, val_groups = [], []

    for label in range(10):
        positions = np.flatnonzero(targets == label)

        if len(positions) != 5000:
            raise RuntimeError('Expected 5,000 training images per CIFAR-10 class.')

        shuffled = rng.permutation(positions)
        train_groups.append(shuffled[:4500])
        val_groups.append(shuffled[4500:])
    train_idx = np.sort(np.concatenate(train_groups))
    val_idx = np.sort(np.concatenate(val_groups))

    with SPLIT_FILE.open('wb') as handle:
        np.savez_compressed(handle, train_idx=train_idx, val_idx=val_idx)

if (len(train_idx) != 45000 or len(val_idx) != 5000
        or len(np.unique(np.concatenate((train_idx, val_idx)))) != 50000
        or np.any(np.bincount(targets[train_idx], minlength=10) != 4500)
        or np.any(np.bincount(targets[val_idx], minlength=10) != 500)):
    raise RuntimeError('Split audit failed; stop before training.')

SPLIT_SHA = hashlib.sha256(SPLIT_FILE.read_bytes()).hexdigest()

print('Training:', len(train_idx), 'Validation:', len(val_idx))
print('Per-class training counts:', np.bincount(targets[train_idx]).tolist())
print('Per-class validation counts:', np.bincount(targets[val_idx]).tolist())
print('Split SHA256:', SPLIT_SHA)

del raw_train

100%|██████████| 170M/170M [38:10<00:00, 74.4kB/s]


Training: 45000 Validation: 5000
Per-class training counts: [4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500]
Per-class validation counts: [500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
Split SHA256: 58e3e0a58ca34ebab457ec178874817e597302512f91228ec13a69420b6164d8


In [5]:
#! Construct loaders without leakage and initialize model/SGD/scheduler
train_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=training_transform(False)), train_idx.tolist()
)

train_clean_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=evaluation_transform()), train_idx.tolist()

)

val_dataset = Subset(
    datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False,
                     transform=evaluation_transform()), val_idx.tolist()
)

train_generator = torch.Generator()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=False, num_workers=NUM_WORKERS,
                          pin_memory=True, generator=train_generator)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

train_clean_loader = DataLoader(train_clean_dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

seed_all(SEED)

model = make_resnet18(10)

if INITIAL_PATH.exists():
    initial = torch.load(INITIAL_PATH, map_location='cpu', weights_only=False)

    if initial['seed'] != SEED or initial['architecture'] != CONFIG['architecture']:
        raise RuntimeError('Shared random initialization is incompatible with this experiment.')

    model.load_state_dict(initial['model_state_dict'], strict=True)

    print('Reusing the existing shared random initialization.')
else:
    atomic_torch_save({
        'seed': SEED, 'architecture': CONFIG['architecture'],
        'model_state_dict': {name: tensor.detach().cpu().clone()
                             for name, tensor in model.state_dict().items()},
    }, INITIAL_PATH)

    print('Saved a random initialization for Vanilla and GCSC.')

INITIAL_SHA = hashlib.sha256(INITIAL_PATH.read_bytes()).hexdigest()
model = model.to(DEVICE)

optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                            weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.)
criterion = torch.nn.CrossEntropyLoss()

print('ResNet-18 trainable parameters:', sum(p.numel() for p in model.parameters()))
print('Initial checkpoint SHA256:', INITIAL_SHA)

Saved a random initialization for Vanilla and GCSC.
ResNet-18 trainable parameters: 11173962
Initial checkpoint SHA256: badc9207a8c7ca962fbc1b7ba972e0652cd2b3b406342b70be7644a0592ab590


In [6]:
#! Sanity-check feature extraction and one float32 backward without updating weights
model.eval()
synthetic = torch.randn(2, 3, 32, 32, device=DEVICE)

with torch.no_grad():
    f, z = forward_features_logits(model, synthetic)
    official = model(synthetic)

if f.shape != (2, 512) or z.shape != (2, 10) or not torch.allclose(z, official, atol=1e-5):
    raise RuntimeError('CIFAR ResNet architecture/feature extraction test failed.')

model.zero_grad(set_to_none=True)
_, z = forward_features_logits(model, synthetic)

probe_loss = F.cross_entropy(z, torch.tensor([0, 1], device=DEVICE))
probe_loss.backward()

if (not bool(torch.isfinite(probe_loss))
        or model.fc.weight.grad is None
        or not bool(torch.isfinite(model.fc.weight.grad).all())):
    raise RuntimeError('Synthetic float32 forward/backward produced non-finite values.')

model.zero_grad(set_to_none=True)

del synthetic, f, z, official, probe_loss

print('CIFAR stem, 512-D representation, logits and gradients: PASS')

CIFAR stem, 512-D representation, logits and gradients: PASS


In [7]:
#! Safe completion-only resumption, accuracy evaluation, and checkpoint metadata
history = []

best_acc, best_epoch, start_epoch = -1.0, 0, 1

if LAST_PATH.exists():
    state = torch.load(LAST_PATH, map_location='cpu', weights_only=False)

    for key, expected in [('config_sha256', CONFIG_SHA), ('split_sha256', SPLIT_SHA),
                          ('initial_sha256', INITIAL_SHA)]:
        if state.get(key) != expected:
            raise RuntimeError('Refusing to resume: checkpoint ' + key + ' differs.')

    if BEST_PATH.exists():
        best_metadata = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
        if best_metadata['epoch'] > state['epoch']:
            raise RuntimeError('Interrupted between best and last checkpoint saves; inspect before rerunning.')

    model.load_state_dict(state['model_state_dict'], strict=True)
    optimizer.load_state_dict(state['optimizer_state_dict'])
    scheduler.load_state_dict(state['scheduler_state_dict'])

    history = state['history']

    best_acc, best_epoch = float(state['best_val_accuracy']), int(state['best_epoch'])

    start_epoch = int(state['epoch']) + 1

    if len(history) != start_epoch - 1 or best_epoch < 1 or not BEST_PATH.exists():
        raise RuntimeError('Saved checkpoint/history/best checkpoint are inconsistent.')

    print('Resuming at epoch:', start_epoch, '| best:', best_epoch, best_acc)
elif BEST_PATH.exists():
    raise RuntimeError('Best checkpoint exists without last checkpoint: inspect before continuing.')
else:
    print('Starting Vanilla training from epoch 1.')

@torch.inference_mode()

def validation_metrics(net, loader):
    net.eval()

    correct, examples, loss_sum = 0, 0, 0.0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = net(images)

        loss = criterion(logits, labels)

        if not bool(torch.isfinite(loss)):
            raise FloatingPointError('Non-finite validation loss. Stop without selecting a checkpoint.')

        correct += int((logits.argmax(dim=1) == labels).sum().item())

        examples += labels.size(0)
        loss_sum += loss.item() * labels.size(0)

    if examples != 5000:
        raise RuntimeError('Validation loader must contain exactly 5,000 examples.')
    return {'accuracy': correct / examples, 'loss': loss_sum / examples}

print('Checkpoint and validation helpers: ready')

Starting Vanilla training from epoch 1.
Checkpoint and validation helpers: ready


In [8]:
#! Roll back any interrupted epoch before training; resume only from complete saved epochs
#! This protects against rerunning this cell after a mid-epoch Colab exception.
if LAST_PATH.exists():
    rollback = torch.load(LAST_PATH, map_location='cpu', weights_only=False)

    if (rollback['config_sha256'] != CONFIG_SHA
            or rollback['split_sha256'] != SPLIT_SHA
            or rollback['initial_sha256'] != INITIAL_SHA):
        raise RuntimeError('Refusing to resume a checkpoint with incompatible experiment metadata.')

    if not BEST_PATH.exists():
        raise RuntimeError('Last checkpoint exists but best checkpoint is missing.')

    current_best = torch.load(BEST_PATH, map_location='cpu', weights_only=False)

    if current_best['epoch'] > rollback['epoch']:
        raise RuntimeError('Best was saved but last was interrupted; inspect checkpoints before rerunning.')

    model.load_state_dict(rollback['model_state_dict'], strict=True)
    optimizer.load_state_dict(rollback['optimizer_state_dict'])
    scheduler.load_state_dict(rollback['scheduler_state_dict'])

    history = list(rollback['history'])
    best_acc, best_epoch = float(rollback['best_val_accuracy']), int(rollback['best_epoch'])
    start_epoch = int(rollback['epoch']) + 1
else:
    if BEST_PATH.exists():
        raise RuntimeError('Best checkpoint exists without a complete last checkpoint.')

    initial = torch.load(INITIAL_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(initial['model_state_dict'], strict=True)

    optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM,
                                weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=0.)

    history, best_acc, best_epoch, start_epoch = [], -1.0, 0, 1
print('Verified complete-epoch starting point:', start_epoch)

#! Train the Vanilla baseline with 100-epoch SGD and cosine decay
if start_epoch > EPOCHS:
    print('All 100 epochs already finished: reusing saved checkpoint.')
else:
    for epoch in range(start_epoch, EPOCHS + 1):
        start = time.perf_counter()

        seed_all(SEED + epoch)
        train_generator.manual_seed(SEED + 1009 * epoch)

        model.train()

        running_loss, running_correct, running_count = 0., 0, 0
        epoch_lr = optimizer.param_groups[0]['lr']

        for step, (images, labels) in enumerate(train_loader):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)

            if not bool(torch.isfinite(loss)):
                raise FloatingPointError(f'Non-finite Vanilla loss at epoch {epoch}, batch {step}.')

            loss.backward()

            if step % 100 == 0:
                if any(p.grad is not None and not bool(torch.isfinite(p.grad).all())
                       for p in model.parameters()):
                    raise FloatingPointError(f'Non-finite Vanilla gradient at epoch {epoch}, batch {step}.')

            optimizer.step()
            count = labels.size(0)

            running_count += count
            running_loss += float(loss.detach()) * count
            running_correct += int((logits.detach().argmax(1) == labels).sum().item())

        if running_count != 45000:
            raise RuntimeError('Each epoch must process exactly 45,000 known training examples.')

        val = validation_metrics(model, val_loader)
        improved = val['accuracy'] > best_acc

        if improved:
            best_acc, best_epoch = val['accuracy'], epoch

            atomic_torch_save({
                'method': 'vanilla', 'epoch': epoch, 'seed': SEED,
                'best_val_accuracy': best_acc,
                'config_sha256': CONFIG_SHA, 'split_sha256': SPLIT_SHA,
                'initial_sha256': INITIAL_SHA,
                'model_state_dict': {k: v.detach().cpu().clone()
                                     for k, v in model.state_dict().items()},
            }, BEST_PATH)

        history.append({
            'epoch': epoch, 'learning_rate': epoch_lr,
            'train_loss': running_loss / running_count,
            'train_accuracy': running_correct / running_count,
            'val_loss': val['loss'], 'val_accuracy': val['accuracy'],
            'best_val_accuracy': best_acc, 'best_epoch': best_epoch,
            'elapsed_seconds': time.perf_counter() - start,
        })
        scheduler.step()

        atomic_torch_save({
            'method': 'vanilla', 'epoch': epoch, 'finished': epoch == EPOCHS,
            'model_state_dict': {k: v.detach().cpu().clone()
                                 for k, v in model.state_dict().items()},
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history': history, 'best_val_accuracy': best_acc,
            'best_epoch': best_epoch, 'config_sha256': CONFIG_SHA,
            'split_sha256': SPLIT_SHA, 'initial_sha256': INITIAL_SHA,
        }, LAST_PATH)

        pd.DataFrame(history).to_csv(HISTORY_FILE, index=False)

        print(f'Epoch {epoch:03d}/{EPOCHS} | LR {epoch_lr:.6f} | '
              f'train loss {history[-1]["train_loss"]:.4f} | '
              f'train acc {history[-1]["train_accuracy"]:.4f} | '
              f'val acc {val["accuracy"]:.4f} | best {best_acc:.4f} '
              f'(epoch {best_epoch}) | {history[-1]["elapsed_seconds"]:.1f}s')

print('Completed epochs:', len(history), '/', EPOCHS,
      '| best epoch:', best_epoch, '| best validation accuracy:', round(best_acc, 5))

Verified complete-epoch starting point: 1
Epoch 001/100 | LR 0.100000 | train loss 2.0284 | train acc 0.2588 | val acc 0.4038 | best 0.4038 (epoch 1) | 36.6s
Epoch 002/100 | LR 0.099975 | train loss 1.4913 | train acc 0.4465 | val acc 0.4942 | best 0.4942 (epoch 2) | 37.4s
Epoch 003/100 | LR 0.099901 | train loss 1.1979 | train acc 0.5669 | val acc 0.5888 | best 0.5888 (epoch 3) | 37.9s
Epoch 004/100 | LR 0.099778 | train loss 0.9657 | train acc 0.6581 | val acc 0.6518 | best 0.6518 (epoch 4) | 38.5s
Epoch 005/100 | LR 0.099606 | train loss 0.7944 | train acc 0.7198 | val acc 0.6854 | best 0.6854 (epoch 5) | 38.3s
Epoch 006/100 | LR 0.099384 | train loss 0.6847 | train acc 0.7623 | val acc 0.6990 | best 0.6990 (epoch 6) | 38.5s
Epoch 007/100 | LR 0.099114 | train loss 0.6070 | train acc 0.7907 | val acc 0.7620 | best 0.7620 (epoch 7) | 38.4s
Epoch 008/100 | LR 0.098796 | train loss 0.5620 | train acc 0.8069 | val acc 0.7926 | best 0.7926 (epoch 8) | 38.4s
Epoch 009/100 | LR 0.098429 | 

In [9]:
#! Extract only unaugmented CIFAR-10 train/validation outputs from frozen best model
if not LAST_PATH.exists() or not BEST_PATH.exists():
    raise RuntimeError('Finish the 100-epoch Vanilla run before feature extraction.')

last_meta = torch.load(LAST_PATH, map_location='cpu', weights_only=False)

if last_meta['epoch'] != EPOCHS or not last_meta['finished']:
    raise RuntimeError('Vanilla 100 epochs are incomplete; do not extract partial-model results.')

best_meta = torch.load(BEST_PATH, map_location='cpu', weights_only=False)

if any(best_meta[key] != value for key, value in [
        ('config_sha256', CONFIG_SHA), ('split_sha256', SPLIT_SHA),
        ('initial_sha256', INITIAL_SHA)]):
    raise RuntimeError('Selected best checkpoint metadata differs from the current protocol.')

BEST_SHA = hashlib.sha256(BEST_PATH.read_bytes()).hexdigest()
model.load_state_dict(best_meta['model_state_dict'], strict=True)

model.eval()

for name, indices, loader in [
    ('known_train', train_idx, train_clean_loader),
    ('known_val', val_idx, val_loader),
]:
    output_file = CACHE_DIR / f'{name}_features_logits.npz'
    if output_file.exists():
        with np.load(output_file) as cached:
            if (str(cached['best_checkpoint_sha256'].item()) != BEST_SHA
                    or not np.array_equal(cached['indices'], indices)
                    or cached['features'].shape != (len(indices), 512)
                    or cached['logits'].shape != (len(indices), 10)):
                raise RuntimeError('Existing output cache differs; inspect before overwriting: ' + str(output_file))
        print('Reusing:', name)
        continue

    features, logits, labels = extract_known_outputs(model, loader, DEVICE)

    if (features.shape != (len(indices), 512)
            or logits.shape != (len(indices), 10)
            or not np.array_equal(labels, targets[indices])
            or not np.isfinite(features).all() or not np.isfinite(logits).all()):
        raise RuntimeError('Known-output extraction failed its alignment or finite-value checks.')

    temporary = output_file.with_name(output_file.name + '.tmp')

    with temporary.open('wb') as handle:
        np.savez_compressed(handle, features=features, logits=logits,
                            labels=labels, indices=indices,
                            best_checkpoint_sha256=np.array(BEST_SHA))

    os.replace(temporary, output_file)
    print('Saved:', name, 'features:', features.shape, 'logits:', logits.shape)

print('Known-only extraction complete; unknown/test outputs reserved for final evaluation.')

Saved: known_train features: (45000, 512) logits: (45000, 10)
Saved: known_val features: (5000, 512) logits: (5000, 10)
Known-only extraction complete; unknown/test outputs reserved for final evaluation.


In [10]:
#! Verify final Vanilla checkpoint, per-epoch history, and feature caches
last = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
best = torch.load(BEST_PATH, map_location='cpu', weights_only=False)

if last['epoch'] != EPOCHS or not last['finished'] or len(last['history']) != EPOCHS:
    raise RuntimeError('Vanilla training has not finished all 100 epochs.')

if best['epoch'] != last['best_epoch'] or best['best_val_accuracy'] != last['best_val_accuracy']:
    raise RuntimeError('Best checkpoint and training history disagree.')

for name, idx in [('known_train', train_idx), ('known_val', val_idx)]:
    path = CACHE_DIR / f'{name}_features_logits.npz'

    with np.load(path) as saved:
        if (not np.array_equal(saved['indices'], idx)
                or not np.array_equal(saved['labels'], targets[idx])
                or saved['features'].shape != (len(idx), 512)
                or saved['logits'].shape != (len(idx), 10)
                or str(saved['best_checkpoint_sha256'].item()) != BEST_SHA):
            raise RuntimeError('Cache audit failed: ' + name)

print('TASK 4A COMPLETE')
print('Best CIFAR-10 validation accuracy:', round(best['best_val_accuracy'], 5),
      '| selected epoch:', best['epoch'])
print('Vanilla best:', BEST_PATH)
print('Shared random initialization for GCSC:', INITIAL_PATH)
print('Known feature caches:', CACHE_DIR)
print('CIFAR-10 TEST AND ALL CIFAR-100 EXAMPLES HAVE NOT BEEN ACCESSED.')

TASK 4A COMPLETE
Best CIFAR-10 validation accuracy: 0.95 | selected epoch: 95
Vanilla best: /content/drive/MyDrive/pa1-beyond-iid/task4/checkpoints/vanilla/best.pt
Shared random initialization for GCSC: /content/drive/MyDrive/pa1-beyond-iid/task4/checkpoints/shared_random_resnet18_seed6304.pt
Known feature caches: /content/drive/MyDrive/pa1-beyond-iid/task4/cache/vanilla
CIFAR-10 TEST AND ALL CIFAR-100 EXAMPLES HAVE NOT BEEN ACCESSED.
